# SST Indices Preprocessing

This notebook runs the generalized SST index preprocessing workflow. It calls `scripts/run_process_sst_index.py` to:
1. Load monthly SST inputs for E3SM, observations (HadISST2), and CESM-SMYLE.
2. Compute monthly and seasonal SST indices, including Nino regions, TNA, TSA, IOD, TNI, ONI, RONI, and Atlantic indices.
3. Compute regridded ELI for observations, CESM-SMYLE, and optionally E3SM, plus native-grid ELI from E3SM MPAS-Ocean history files.
4. Save SST-index time-series outputs into the clean, model-first `sst_index/timeseries` layout used by the downstream diagnostics.

Outputs are written under:
- `JRA55_FOSIRL/sst_index/timeseries` and `Reanalysis/sst_index/timeseries` for E3SM cases.
- `HadISST2/sst_index/timeseries` for observations.
- `CESM-SMYLE/sst_index/timeseries` for the CESM-SMYLE benchmark.

The ELI section is included because ELI uses full-field centroid logic rather than a rectangular regional mean. E3SM can write native, regridded, or both ELI product sets.


In [12]:
import os
import subprocess
import sys
from pathlib import Path

# Identify repository root
REPO_ROOT = Path(os.getcwd())
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = Path("..")

In [13]:
# -----------------------------
# Configuration
# -----------------------------
# Multi-case E3SM hindcasts. Add more entries here as new post-processed
# hindcasts land in different data directories.
E3SM_CASES = {
    "E3SM-FOSIRL": {
        "data_dir": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
        "eli_data_dir": "/global/cfs/cdirs/e3smdata/simulations/S2S2D",
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "cache_tag": "JRA55_FOSIRL",
        "display_name": "E3SMv3-FOSIRL",
    },
    "E3SM-Reanalysis": {
        "data_dir": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
        "eli_data_dir": "/global/cfs/cdirs/e3sm/S2S2D/simulation",
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "cache_tag": "Reanalysis",
        "display_name": "E3SMv3-Reanalysis",
    },
#    "E3SM-4DEnVarOcn": {
#        "data_dir": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
#        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
#        "cache_tag": "4DEnVarOcn",
#        "display_name": "E3SMv3-4DEnVarOcn",
#    },
}

CONFIG = {
    "sources": ["obs", "e3sm", "smyle"],
    "e3sm_cases": E3SM_CASES,
    "regions": [
        "IOD", "TNI", "ONI", "RONI",
        "Nino12", "Nino3", "Nino3.4", "Nino4",
        "TNA", "TSA", "PACWRAMPOOL", "AtlNino", "AtlMDR"
    ],  # List of target indices/regions to compute
    "custom_regions": {
        "Nino12": {
            "lonlat": [270.0, 280.0, -10.0, 0.0],
            "long_name": "Nino 1+2 regional mean SST",
        },
        "Nino3": {
            "lonlat": [210.0, 270.0, -5.0, 5.0],
            "long_name": "Nino 3 regional mean SST",
        },
        "Nino3.4": {
            "lonlat": [190.0, 240.0, -5.0, 5.0],
            "long_name": "Nino 3.4 regional mean SST",
        },
        "Nino4": {
            "lonlat": [160.0, 210.0, -5.0, 5.0],
            "long_name": "Nino 4 regional mean SST",
        },
        "TNA": {
            "lonlat": [305.0, 345.0, 5.0, 25.0],
            "long_name": "TNA regional mean SST",
        },
        "TSA": {
            "lonlat": [330.0, 10.0, -20.0, 0.0],
            "long_name": "TSA regional mean SST",
        },
        "PACWRAMPOOL": {
            "lonlat": [60.0, 170.0, -15.0, 15.0],
            "long_name": "PACWRAMPOOL regional mean SST",
        },
        "AtlNino": {
            "lonlat": [340.0, 360.0, -3.0, 3.0],
            "long_name": "Atlantic Nino regional mean SST",
        },
        "AtlMDR": {
            "lonlat": [280.0, 350.0, 10.0, 20.0],
            "long_name": "Atlantic MDR regional mean SST",
        },
        # Helper regions for derived indices
        "IOD_West": {
            "lonlat": [50.0, 70.0, -10.0, 10.0],
            "long_name": "IOD West regional mean SST",
        },
        "IOD_East": {
            "lonlat": [90.0, 110.0, -10.0, 0.0],
            "long_name": "IOD East regional mean SST",
        },
        "TropicalMean": {
            "lonlat": [0.0, 360.0, -20.0, 20.0],
            "long_name": "Tropical Mean regional mean SST",
        },
    },
    "outdir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag",
    "smyle_outdir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE",
    "obs_outdir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/HadISST2/sst_index/timeseries",
    "init_months": [5, 11],
    "year_start": 1980,
    "year_end": 2018,
    "climy0": 1981,
    "climy1": 2010,
    "nlead": 24,
    "e3sm_nens": 10,
    "smyle_nens": 20,
    "workers": 8,
    "force": False,  # Set to True to force rewrite
    "sst_land_mask": True,  # Apply cached source-grid land masks
    "process_eli": True,
    # Native MPAS-Ocean ELI reads many NetCDF files. Keep this serial by
    # default because netCDF/HDF5 reads are not reliable from Python threads.
    "eli_workers": 1,
    # "native" uses MPAS-Ocean history files, "regridded" uses the same
    # lat/lon TS inputs as the other indices, and "both" writes both E3SM products.
    "eli_input_grid": "both",
    # Generate missing CESM-SMYLE TS monthly/seasonal benchmark inputs before
    # computing CESM-SMYLE SST indices. Existing files are not overwritten.
    "auto_generate_smyle_benchmark": True,
}


In [14]:
# -----------------------------
# Construct and execute command for each index/case
# -----------------------------
import json

script_path = str(REPO_ROOT / "scripts" / "run_process_sst_index.py")

# Set environment variables for GDAL/PROJ
env = os.environ.copy()
conda_prefix = "/global/homes/z/zhan391/.conda/envs/e3sm_analysis"
env["GDAL_DATA"] = f"{conda_prefix}/share/gdal"
env["PROJ_LIB"] = f"{conda_prefix}/share/proj"


def build_base_cmd(sources, regions):
    cmd = [
        sys.executable,
        script_path,
        "--sources", *sources,
        "--outdir", CONFIG["outdir"],
        "--smyle-outdir", CONFIG["smyle_outdir"],
        "--obs-outdir", CONFIG["obs_outdir"],
        "--init-months", *(str(m) for m in CONFIG["init_months"]),
        "--year-start", str(CONFIG["year_start"]),
        "--year-end", str(CONFIG["year_end"]),
        "--climy0", str(CONFIG["climy0"]),
        "--climy1", str(CONFIG["climy1"]),
        "--nlead", str(CONFIG["nlead"]),
        "--e3sm-nens", str(CONFIG["e3sm_nens"]),
        "--smyle-nens", str(CONFIG["smyle_nens"]),
        "--workers", str(CONFIG["workers"]),
        "--regions", *regions,
    ]

    if CONFIG.get("custom_regions"):
        cmd.extend(["--custom-regions", json.dumps(CONFIG["custom_regions"])])

    if CONFIG["force"]:
        cmd.append("--force")
    if not CONFIG.get("sst_land_mask", True):
        cmd.append("--no-sst-land-mask")

    return cmd


def run_cmd(cmd, label, regions):
    print("=" * 60)
    print(f"Processing SST indices: {', '.join(regions)} | {label}")
    print("=" * 60)
    print("Running command:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, env=env, text=True)

    if result.returncode != 0:
        raise RuntimeError(
            f"Preprocessing failed for '{label}' indices {regions} "
            f"with exit code {result.returncode}"
        )




def smyle_benchmark_files():
    root = Path(CONFIG["smyle_outdir"]) / "leadtime_acc" / "inputs" / "TS"
    files = []
    for init_month in CONFIG["init_months"]:
        for freq in ("mon", "seas"):
            files.append(
                root / f"BSMYLE{init_month:02d}_TS_N{CONFIG['smyle_nens']:02d}_M{CONFIG['nlead']:02d}_{freq}.nc"
            )
    return files


def build_smyle_benchmark_cmd():
    return [
        sys.executable,
        str(REPO_ROOT / "scripts" / "run_process_cesm_smyle_benchmark.py"),
        "--fields", "TS",
        "--init-months", *(str(m) for m in CONFIG["init_months"]),
        "--year-start", str(CONFIG["year_start"]),
        "--year-end", str(CONFIG["year_end"]),
        "--nens", str(CONFIG["smyle_nens"]),
        "--nlead", str(CONFIG["nlead"]),
        "--workers", str(CONFIG["workers"]),
        "--freqs", "mon", "seas",
        "--outdir", CONFIG["smyle_outdir"],
    ]


def ensure_smyle_benchmark_inputs():
    if "smyle" not in CONFIG["sources"]:
        return

    missing = [path for path in smyle_benchmark_files() if not path.exists()]
    if not missing:
        print("CESM-SMYLE TS benchmark inputs already exist.")
        return

    cmd = build_smyle_benchmark_cmd()
    if not CONFIG.get("auto_generate_smyle_benchmark", False):
        missing_text = "\n".join(f"  {path}" for path in missing)
        raise FileNotFoundError(
            "Missing CESM-SMYLE TS benchmark inputs needed for SST indices:\n"
            f"{missing_text}\n\n"
            "Generate them with:\n"
            + " ".join(cmd)
        )

    print("Missing CESM-SMYLE TS benchmark inputs; generating them now:")
    print(" ".join(cmd))
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)
    print("\n--- SMYLE BENCHMARK STDOUT ---")
    print(result.stdout)
    if result.returncode != 0:
        print("\n--- SMYLE BENCHMARK STDERR ---")
        print(result.stderr)
        raise RuntimeError(
            f"CESM-SMYLE TS benchmark generation failed with exit code {result.returncode}"
        )

    still_missing = [path for path in smyle_benchmark_files() if not path.exists()]
    if still_missing:
        missing_text = "\n".join(f"  {path}" for path in still_missing)
        raise FileNotFoundError(
            "CESM-SMYLE benchmark generation completed, but these expected files are still missing:\n"
            f"{missing_text}"
        )


def sst_index_regions_to_process():
    regions = list(CONFIG["regions"])
    if CONFIG.get("process_eli", True) and "ELI" not in regions:
        regions.append("ELI")
    return regions


VALID_ELI_INPUT_GRIDS = {"native", "regridded", "both"}
ELI_INPUT_GRID = CONFIG.get("eli_input_grid", "native")
if ELI_INPUT_GRID not in VALID_ELI_INPUT_GRIDS:
    raise ValueError(
        f"Unsupported eli_input_grid={ELI_INPUT_GRID!r}; "
        f"choose one of {sorted(VALID_ELI_INPUT_GRIDS)}."
    )


def should_process_e3sm_region(region):
    if region == "ELI":
        return CONFIG.get("process_eli", True) and ELI_INPUT_GRID in {"regridded", "both"}
    return True


ensure_smyle_benchmark_inputs()

shared_sources = [s for s in CONFIG["sources"] if s != "e3sm"]
process_e3sm = "e3sm" in CONFIG["sources"]

all_regions = sst_index_regions_to_process()
for shared_source in shared_sources:
    cmd = build_base_cmd([shared_source], all_regions)
    run_cmd(cmd, shared_source, all_regions)

e3sm_regions = [r for r in all_regions if should_process_e3sm_region(r)]
if process_e3sm and e3sm_regions:
    for case_key, case_info in CONFIG["e3sm_cases"].items():
        cmd = build_base_cmd(["e3sm"], e3sm_regions)
        cmd.extend([
            "--e3sm-data-dir", case_info["data_dir"],
            "--e3sm-case-prefix", case_info["case_prefix"],
            "--e3sm-cache-tag", case_info["cache_tag"],
            "--e3sm-display-name", case_info.get("display_name", case_key),
        ])
        run_cmd(cmd, case_key, e3sm_regions)

print("\nAll SST indices processed successfully!")


CESM-SMYLE TS benchmark inputs already exist.
Processing SST indices: IOD, TNI, ONI, RONI, Nino12, Nino3, Nino3.4, Nino4, TNA, TSA, PACWRAMPOOL, AtlNino, AtlMDR, ELI | obs
Running command:
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python ../scripts/run_process_sst_index.py --sources obs --outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag --smyle-outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE --obs-outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/HadISST2/sst_index/timeseries --init-months 5 11 --year-start 1980 --year-end 2018 --climy0 1981 --climy1 2010 --nlead 24 --e3sm-nens 10 --smyle-nens 20 --workers 8 --regions IOD TNI ONI RONI Nino12 Nino3 Nino3.4 Nino4 TNA TSA PACWRAMPOOL AtlNino AtlMDR ELI --custom-regions {"Nino12": {"lonlat": [270.0, 280.0, -10.0, 0.0], "long_name": "Nino 1+2 regional mean SST"}, "Nino3": {"lonlat": [210.0, 270.0, -5.0, 5.0], "long_name": "Nino 3 regional mean SST"}, "Nino3.4": {"lonlat": [190.0, 240.0, -5.0, 5.0], "long_name": "Nino 3.4 regional

/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44405 instead
  warnings.warn(
13:22:19 - __main__ - INFO - Dask Dashboard: http://127.0.0.1:44405/status
13:22:19 - __main__ - INFO - Processing Observations (HadISST2) regional SST Indices...
13:22:22 - __main__ - INFO - Observed SST land mask enabled: True
13:22:22 - __main__ - INFO - Computing Observations base indices...
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 17.65 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
13:22:

Processing SST indices: IOD, TNI, ONI, RONI, Nino12, Nino3, Nino3.4, Nino4, TNA, TSA, PACWRAMPOOL, AtlNino, AtlMDR, ELI | smyle
Running command:
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python ../scripts/run_process_sst_index.py --sources smyle --outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag --smyle-outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE --obs-outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/HadISST2/sst_index/timeseries --init-months 5 11 --year-start 1980 --year-end 2018 --climy0 1981 --climy1 2010 --nlead 24 --e3sm-nens 10 --smyle-nens 20 --workers 8 --regions IOD TNI ONI RONI Nino12 Nino3 Nino3.4 Nino4 TNA TSA PACWRAMPOOL AtlNino AtlMDR ELI --custom-regions {"Nino12": {"lonlat": [270.0, 280.0, -10.0, 0.0], "long_name": "Nino 1+2 regional mean SST"}, "Nino3": {"lonlat": [210.0, 270.0, -5.0, 5.0], "long_name": "Nino 3 regional mean SST"}, "Nino3.4": {"lonlat": [190.0, 240.0, -5.0, 5.0], "long_name": "Nino 3.4 regional mean SST"}, "Nino4": {"lonlat": [160.0, 2

/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35195 instead
  warnings.warn(
13:22:39 - __main__ - INFO - Dask Dashboard: http://127.0.0.1:35195/status
13:22:39 - __main__ - INFO - Processing CESM-SMYLE regional SST Indices...
13:22:41 - __main__ - INFO - Computing CESM-SMYLE base indices for month 5...
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 15.54 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packa

Processing SST indices: IOD, TNI, ONI, RONI, Nino12, Nino3, Nino3.4, Nino4, TNA, TSA, PACWRAMPOOL, AtlNino, AtlMDR, ELI | E3SM-FOSIRL
Running command:
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python ../scripts/run_process_sst_index.py --sources e3sm --outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag --smyle-outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE --obs-outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/HadISST2/sst_index/timeseries --init-months 5 11 --year-start 1980 --year-end 2018 --climy0 1981 --climy1 2010 --nlead 24 --e3sm-nens 10 --smyle-nens 20 --workers 8 --regions IOD TNI ONI RONI Nino12 Nino3 Nino3.4 Nino4 TNA TSA PACWRAMPOOL AtlNino AtlMDR ELI --custom-regions {"Nino12": {"lonlat": [270.0, 280.0, -10.0, 0.0], "long_name": "Nino 1+2 regional mean SST"}, "Nino3": {"lonlat": [210.0, 270.0, -5.0, 5.0], "long_name": "Nino 3 regional mean SST"}, "Nino3.4": {"lonlat": [190.0, 240.0, -5.0, 5.0], "long_name": "Nino 3.4 regional mean SST"}, "Nino4": {"lonlat": [160

13:24:25 - __main__ - INFO - Dask Dashboard: http://127.0.0.1:8787/status
13:24:25 - __main__ - INFO - Processing E3SM regional SST Indices for E3SMv3-FOSIRL...
13:25:24 - __main__ - INFO - E3SM SST land mask enabled: True
13:25:27 - __main__ - INFO - Computing base indices for month 5...
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 31.56 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages

Processing SST indices: IOD, TNI, ONI, RONI, Nino12, Nino3, Nino3.4, Nino4, TNA, TSA, PACWRAMPOOL, AtlNino, AtlMDR, ELI | E3SM-Reanalysis
Running command:
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python ../scripts/run_process_sst_index.py --sources e3sm --outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag --smyle-outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE --obs-outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/HadISST2/sst_index/timeseries --init-months 5 11 --year-start 1980 --year-end 2018 --climy0 1981 --climy1 2010 --nlead 24 --e3sm-nens 10 --smyle-nens 20 --workers 8 --regions IOD TNI ONI RONI Nino12 Nino3 Nino3.4 Nino4 TNA TSA PACWRAMPOOL AtlNino AtlMDR ELI --custom-regions {"Nino12": {"lonlat": [270.0, 280.0, -10.0, 0.0], "long_name": "Nino 1+2 regional mean SST"}, "Nino3": {"lonlat": [210.0, 270.0, -5.0, 5.0], "long_name": "Nino 3 regional mean SST"}, "Nino3.4": {"lonlat": [190.0, 240.0, -5.0, 5.0], "long_name": "Nino 3.4 regional mean SST"}, "Nino4": {"lonlat": 

13:30:11 - __main__ - INFO - Dask Dashboard: http://127.0.0.1:8787/status
13:30:11 - __main__ - INFO - Processing E3SM regional SST Indices for E3SMv3-Reanalysis...
13:31:12 - __main__ - INFO - E3SM SST land mask enabled: True
13:31:16 - __main__ - INFO - Computing base indices for month 5...
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 31.56 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/lib/python3.14/site-pack


All SST indices processed successfully!


## ELI Index Preprocessing

The regular command path writes regridded ELI for observations and CESM-SMYLE and, when selected, E3SM. This section adds E3SM native MPAS-Ocean ELI. With `eli_input_grid="both"`, E3SM receives distinct monthly and seasonal native and regridded product files in the same `sst_index/timeseries` directory.

In [15]:
# ELI imports and native-library paths.
_env_prefix = sys.prefix
_proj_path = os.path.join(_env_prefix, "share", "proj")
if os.path.isfile(os.path.join(_proj_path, "proj.db")):
    os.environ["CONDA_PREFIX"] = _env_prefix
    os.environ["PROJ_LIB"] = _proj_path
    os.environ["PROJ_DATA"] = _proj_path

import warnings
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import cftime
import numpy as np
import xarray as xr
from esp_lab.utils.sst_utils import (
    SST_PREPROCESSING_VERSION, mask_nonphysical_sst, normalize_sst_to_degc,
)

print(f"Python      : {sys.executable}")
print(f"CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', '(not set)')}")


Python      : /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python
CONDA_PREFIX: /global/homes/z/zhan391/.conda/envs/e3sm_analysis


In [16]:
# ------------------------------------------------------------------ #
#  ELI configuration
# ------------------------------------------------------------------ #

ELI_INPUT_GRID = CONFIG.get("eli_input_grid", "native")
if ELI_INPUT_GRID not in {"native", "regridded", "both"}:
    raise ValueError(f"Unsupported eli_input_grid: {ELI_INPUT_GRID!r}")
PROCESS_NATIVE_ELI = (
    CONFIG.get("process_eli", True)
    and ELI_INPUT_GRID in {"native", "both"}
)
S2D_DIAG_ROOT = Path(CONFIG["outdir"])

# MPAS-Ocean mesh file for E3SM native-ocean ELI.
MESH_FILE = Path(
    "/global/cfs/cdirs/e3sm/inputdata/ocn/mpas-o/IcoswISC30E3r5/"
    "mpaso.IcoswISC30E3r5.rstFromG-chrysalis.20231121.nc"
)

# ELI uses native MPAS-Ocean history files, not the post-processed atm files.
E3SM_CASES = {}
if "e3sm" in CONFIG["sources"]:
    for case_key, case_info in CONFIG["e3sm_cases"].items():
        E3SM_CASES[case_key] = {
            **case_info,
            "data_dir": case_info.get("eli_data_dir", case_info["data_dir"]),
        }

START_YEAR = int(CONFIG["year_start"])
END_YEAR = int(CONFIG["year_end"])
EXCL_YEAR = None
INIT_MONTHS = list(CONFIG["init_months"])
lead_years = [y for y in np.arange(START_YEAR, END_YEAR + 1) if y != EXCL_YEAR]

CASES_BY_E3SM_CASE = {}
CASE_TO_DATA_DIR = {}
OUTDIR_BY_E3SM_CASE = {}
for case_key, case_info in E3SM_CASES.items():
    data_dir = Path(case_info["data_dir"])
    case_prefix = case_info["case_prefix"]
    cache_tag = case_info["cache_tag"]
    cases = [
        f"{case_prefix}_{year}{init_month:02d}0100"
        for init_month in INIT_MONTHS
        for year in lead_years
    ]
    CASES_BY_E3SM_CASE[case_key] = cases
    OUTDIR_BY_E3SM_CASE[case_key] = S2D_DIAG_ROOT / cache_tag / "sst_index" / "timeseries"
    for case in cases:
        CASE_TO_DATA_DIR[case] = data_dir

CASES = [case for cases in CASES_BY_E3SM_CASE.values() for case in cases]

CASE_NENS = int(CONFIG["e3sm_nens"])
MEMBERS = [f"EN{i:02d}" for i in range(CASE_NENS)]
NENS = None

ELI_LAT_MIN = -5.0
ELI_LAT_MAX = 5.0
ELI_LON_MIN = 120.0
ELI_LON_MAX = 290.0
TC_LAT_HALF = 5.0

OCN_HIST_PATTERN = "*mpaso.hist.am.timeSeriesStatsMonthly.*.nc"
OCN_SST_VAR = "timeMonthly_avg_activeTracers_temperature"

NLEAD = int(CONFIG["nlead"])
FORCE_REWRITE = bool(CONFIG["force"])
NWORKERS = max(1, int(CONFIG.get("eli_workers", 1)))

print(f"ELI_INPUT_GRID: {ELI_INPUT_GRID}")
print(f"PROCESS_NATIVE_ELI: {PROCESS_NATIVE_ELI}")
print(f"NLEAD         : {NLEAD}")
print(f"CASE_NENS     : {CASE_NENS}")
print(f"FORCE_REWRITE : {FORCE_REWRITE}")
print(f"NWORKERS      : {NWORKERS}")
print("CESM-SMYLE/obs ELI: handled by the regular regridded SST-index command path")
for case_key, cases in CASES_BY_E3SM_CASE.items():
    case_info = E3SM_CASES[case_key]
    print(f"  {case_key}: {len(cases)} cases")
    print(f"    data_dir : {case_info['data_dir']}")
    print(f"    outdir   : {OUTDIR_BY_E3SM_CASE[case_key]}")


ELI_INPUT_GRID: both
PROCESS_NATIVE_ELI: True
NLEAD         : 24
CASE_NENS     : 10
FORCE_REWRITE : False
NWORKERS      : 1
CESM-SMYLE/obs ELI: handled by the regular regridded SST-index command path
  E3SM-FOSIRL: 78 cases
    data_dir : /global/cfs/cdirs/e3smdata/simulations/S2S2D
    outdir   : /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/JRA55_FOSIRL/sst_index/timeseries
  E3SM-Reanalysis: 78 cases
    data_dir : /global/cfs/cdirs/e3sm/S2S2D/simulation
    outdir   : /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/Reanalysis/sst_index/timeseries


### Validate ELI Inputs

In [17]:
if not PROCESS_NATIVE_ELI:
    print("Native E3SM ELI is disabled; skipping native-input validation.")
else:
    # ---- enforce all required fields are set ----
    _required = {
        "S2D_DIAG_ROOT":       S2D_DIAG_ROOT,
        "E3SM_CASES":          E3SM_CASES,
        "CASES_BY_E3SM_CASE":  CASES_BY_E3SM_CASE,
        "OUTDIR_BY_E3SM_CASE": OUTDIR_BY_E3SM_CASE,
        "MESH_FILE":           MESH_FILE,
        "CASES":               CASES,
        "MEMBERS":             MEMBERS,
    }
    _missing = [k for k, v in _required.items() if v is None]
    if _missing:
        raise ValueError(
            "The following required fields are still None - fill them in the "
            "Configuration cell:\n" + "\n".join(f"  {k}" for k in _missing)
        )

    # ---- basic configuration checks ----
    _errors = []
    if not E3SM_CASES:
        _errors.append("E3SM_CASES is empty.")
    if not CASES:
        _errors.append("CASES is empty; check START_YEAR, END_YEAR, EXCL_YEAR, and INIT_MONTHS.")
    if not lead_years:
        _errors.append("lead_years is empty; check START_YEAR, END_YEAR, and EXCL_YEAR.")
    if NLEAD <= 0:
        _errors.append(f"NLEAD must be positive, got {NLEAD}.")
    if CASE_NENS <= 0:
        _errors.append(f"CASE_NENS must be positive, got {CASE_NENS}.")
    if NENS is not None and NENS <= 0:
        _errors.append(f"NENS must be positive or None, got {NENS}.")
    if NWORKERS <= 0:
        _errors.append(f"NWORKERS must be positive, got {NWORKERS}.")
    if not MESH_FILE.is_file():
        _errors.append(f"MESH_FILE does not exist: {MESH_FILE}")

    # ---- check paths and reference members for each E3SM case group ----
    members_by_case = {}
    for case_key, case_info in E3SM_CASES.items():
        data_dir = Path(case_info["data_dir"])
        case_names = CASES_BY_E3SM_CASE.get(case_key, [])
        if not data_dir.is_dir():
            _errors.append(f"data_dir does not exist for {case_key}: {data_dir}")
            continue
        if not case_names:
            _errors.append(f"No cases generated for {case_key}.")
            continue

        ref_case_dir = data_dir / case_names[0]
        if not ref_case_dir.is_dir():
            _errors.append(f"First case directory does not exist for {case_key}: {ref_case_dir}")
            continue

        all_members = sorted(p.name for p in ref_case_dir.iterdir()
                             if p.is_dir() and p.name.startswith("EN"))
        if not all_members:
            _errors.append(f"No EN* member directories found in {ref_case_dir}")
            continue

        if MEMBERS:
            missing_members = sorted(set(MEMBERS) - set(all_members))
            if missing_members:
                _errors.append(
                    "Requested member directories are missing in the reference case "
                    f"{ref_case_dir}: {missing_members}"
                )
                continue
            selected = list(MEMBERS)
        else:
            selected = all_members

        if NENS is not None:
            if NENS > len(selected):
                _errors.append(f"NENS={NENS} exceeds available members for {case_key} ({len(selected)}).")
                continue
            selected = selected[:NENS]

        if not selected:
            _errors.append(f"No members selected for {case_key}.")
            continue
        members_by_case[case_key] = selected

    if _errors:
        raise ValueError("Configuration validation failed:\n" + "\n".join(_errors))

    for outdir in OUTDIR_BY_E3SM_CASE.values():
        outdir.mkdir(parents=True, exist_ok=True)
        if not os.access(outdir, os.W_OK):
            raise PermissionError(f"Output directory is not writable: {outdir}")

    # Use the requested/available member list from the first configured group as the
    # output ensemble convention. All enabled groups should use the same member set.
    _first_case_key = next(iter(members_by_case))
    members_avail = members_by_case[_first_case_key]
    RUN_NENS = len(members_avail)
    OUT_NENS = RUN_NENS

    for case_key, selected in members_by_case.items():
        if selected != members_avail:
            raise ValueError(
                f"Member selection differs for {case_key}: {selected}; expected {members_avail}."
            )

    print("Configuration valid")
    print(f"  S2D_DIAG_ROOT : {S2D_DIAG_ROOT}")
    print(f"  MESH_FILE     : {MESH_FILE}")
    for case_key, cases in CASES_BY_E3SM_CASE.items():
        print(f"  {case_key}:")
        print(f"    data_dir : {E3SM_CASES[case_key]['data_dir']}")
        print(f"    outdir   : {OUTDIR_BY_E3SM_CASE[case_key]}")
        print(f"    cases    : {len(cases)} total  ({cases[0]}  ...  {cases[-1]})")
    print(f"  Members       : {members_avail}  (RUN_NENS={RUN_NENS}, NENS={NENS})")


Configuration valid
  S2D_DIAG_ROOT : /global/cfs/cdirs/e3sm/S2S2D/s2d_diag
  MESH_FILE     : /global/cfs/cdirs/e3sm/inputdata/ocn/mpas-o/IcoswISC30E3r5/mpaso.IcoswISC30E3r5.rstFromG-chrysalis.20231121.nc
  E3SM-FOSIRL:
    data_dir : /global/cfs/cdirs/e3smdata/simulations/S2S2D
    outdir   : /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/JRA55_FOSIRL/sst_index/timeseries
    cases    : 78 total  (WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_2018110100)
  E3SM-Reanalysis:
    data_dir : /global/cfs/cdirs/e3sm/S2S2D/simulation
    outdir   : /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/Reanalysis/sst_index/timeseries
    cases    : 78 total  (WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce_1980050100  ...  WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce_2018110100)
  Members       : ['EN00', 'EN01', 'EN02', 'EN03', 'EN04', 'EN05', 'EN06', 'EN07', 'EN08', 'EN09']  (RUN_NENS=10, NENS=None)


### Load MPAS-Ocean Mesh Geometry

In [18]:
if not PROCESS_NATIVE_ELI:
    print("Native E3SM ELI is disabled; skipping mesh setup.")
else:
    with xr.open_dataset(MESH_FILE) as mesh:
        for mesh_var in ["latCell", "lonCell", "areaCell"]:
            if mesh_var not in mesh:
                raise KeyError(f"Required mesh variable {mesh_var!r} is missing from {MESH_FILE}")

        lat  = mesh["latCell"].values  * 180.0 / np.pi   # degrees North
        lon  = mesh["lonCell"].values  * 180.0 / np.pi   # degrees East  [0, 360]
        area = mesh["areaCell"].values                    # m²

    lon = np.mod(lon, 360.0)

    if not (np.isfinite(lat).all() and np.isfinite(lon).all()):
        raise ValueError("Mesh latitude/longitude contains non-finite values.")
    if not (np.isfinite(area).all() and np.all(area > 0.0)):
        raise ValueError("Mesh cell areas must be finite and strictly positive.")

    # Equatorial Pacific region for ELI centroid.
    region_eq = (
        (lat >= ELI_LAT_MIN) & (lat <= ELI_LAT_MAX) &
        (lon >= ELI_LON_MIN) & (lon <= ELI_LON_MAX)
    )

    # Broad tropical band for reference temperature Tc.
    region_tropics = (lat >= -TC_LAT_HALF) & (lat <= TC_LAT_HALF)

    if not np.any(region_eq):
        raise ValueError("Equatorial Pacific mask selected zero mesh cells; check ELI bounds.")
    if not np.any(region_tropics):
        raise ValueError("Tropical reference mask selected zero mesh cells; check TC_LAT_HALF.")

    # Integer indices are faster and cleaner for repeated file reads.
    idx_union = np.flatnonzero(region_eq | region_tropics)
    eq_on_union = region_eq[idx_union]
    tropics_on_union = region_tropics[idx_union]

    lon_eq  = lon[idx_union][eq_on_union].astype(np.float32)
    area_eq = area[idx_union][eq_on_union].astype(np.float64)
    area_tr = area[idx_union][tropics_on_union].astype(np.float64)

    print(f"Mesh cells total    : {len(lat):,}")
    print(f"Equatorial Pacific  : {region_eq.sum():,} cells  "
          f"(lat [{ELI_LAT_MIN}, {ELI_LAT_MAX}]°, lon [{ELI_LON_MIN}, {ELI_LON_MAX}]°)")
    print(f"Tropical band       : {region_tropics.sum():,} cells  (lat ±{TC_LAT_HALF}°)")

Mesh cells total    : 465,044
Equatorial Pacific  : 24,009 cells  (lat [-5.0, 5.0]°, lon [120.0, 290.0]°)
Tropical band       : 42,941 cells  (lat ±5.0°)


### Compute E3SM ELI

In [ ]:
if not PROCESS_NATIVE_ELI:
    print("Native E3SM ELI is disabled; skipping native E3SM ELI.")
else:
    def _weighted_mean_skipna_1d(values: np.ndarray, weights: np.ndarray) -> float:
        """Area-weighted mean for one time slice, ignoring non-finite values."""
        finite = np.isfinite(values)
        if not np.any(finite):
            return np.nan
        denom = np.sum(weights[finite])
        if denom <= 0.0:
            return np.nan
        return float(np.sum(values[finite] * weights[finite]) / denom)


    def _read_surface_sst_on_cells(path: Path, idx_union: np.ndarray) -> np.ndarray:
        """Read one monthly MPAS-Ocean file on the needed cells only."""
        with xr.open_dataset(path, decode_times=False) as ds:
            if OCN_SST_VAR not in ds:
                raise KeyError(f"{OCN_SST_VAR!r} not found in {path}")

            sst_da = ds[OCN_SST_VAR]
            if "nVertLevels" in sst_da.dims:
                sst_da = sst_da.isel(nVertLevels=0)
            if "Time" in sst_da.dims:
                sst_da = sst_da.isel(Time=0)
            if "nCells" not in sst_da.dims:
                raise ValueError(f"{OCN_SST_VAR!r} must have nCells dimension; found {sst_da.dims}")

            sst_da = normalize_sst_to_degc(sst_da)
            sst_da = mask_nonphysical_sst(sst_da)
            sst = sst_da.isel(nCells=idx_union).astype("float32").load().values

        if sst.ndim != 1:
            raise ValueError(f"Expected 1-D SST after slicing {path}, got shape {sst.shape}")
        return sst


    def _eli_from_sst_on_union(sst: np.ndarray) -> np.float32:
        """Compute one ELI value from SST already subset to ``idx_union``."""
        sst_eq = sst[eq_on_union]
        sst_tr = sst[tropics_on_union]
        tc = _weighted_mean_skipna_1d(sst_tr, area_tr)
        if not np.isfinite(tc):
            return np.float32(np.nan)
        warm = np.isfinite(sst_eq) & (sst_eq > tc)
        denominator = np.sum(area_eq[warm])
        if denominator <= 0.0:
            return np.float32(np.nan)
        return np.float32(np.sum(area_eq[warm] * lon_eq[warm]) / denominator)


    def compute_eli_member(
        hist_dir: Path,
        pattern: str,
        idx_union: np.ndarray,
        eq_on_union: np.ndarray,
        tropics_on_union: np.ndarray,
        lon_eq: np.ndarray,
        area_eq: np.ndarray,
        area_tr: np.ndarray,
        nlead: int,
        seasonal_lead_indices: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray]:
        """
        Compute monthly and seasonal ELI for one ensemble member.

        This reads files one-by-one instead of using ``open_mfdataset``. For this
        workflow that is usually faster because each member has a small fixed number
        of monthly files and only one variable/cell subset is needed.
        """
        files = sorted(hist_dir.glob(pattern))
        if not files:
            return (
                np.full(nlead, np.nan, dtype=np.float32),
                np.full(len(seasonal_lead_indices), np.nan, dtype=np.float32),
            )
        if len(files) < nlead:
            print(f"  [WARN] {hist_dir}: only {len(files)} monthly files found; expected {nlead}.")

        sst_by_lead = np.full((nlead, len(idx_union)), np.nan, dtype=np.float32)
        for lead_idx, path in enumerate(files[:nlead]):
            sst_by_lead[lead_idx] = _read_surface_sst_on_cells(path, idx_union)

        monthly = np.array(
            [_eli_from_sst_on_union(sst) for sst in sst_by_lead],
            dtype=np.float32,
        )
        seasonal = np.full(len(seasonal_lead_indices), np.nan, dtype=np.float32)
        for season_idx, center_idx in enumerate(seasonal_lead_indices):
            if center_idx == 0 or center_idx + 1 >= nlead:
                continue
            window = sst_by_lead[center_idx - 1:center_idx + 2]
            complete = np.all(np.isfinite(window), axis=0)
            seasonal_sst = np.full(window.shape[1], np.nan, dtype=np.float32)
            seasonal_sst[complete] = window[:, complete].mean(axis=0)
            seasonal[season_idx] = _eli_from_sst_on_union(seasonal_sst)

        return monthly, seasonal


    def _compute_case_member(args):
        yi, mi, case, member, hist_dir, seasonal_lead_indices = args
        try:
            monthly, seasonal = compute_eli_member(
                hist_dir, OCN_HIST_PATTERN,
                idx_union, eq_on_union, tropics_on_union,
                lon_eq, area_eq, area_tr,
                NLEAD, seasonal_lead_indices,
            )
            return yi, mi, case, member, monthly, seasonal, None
        except Exception as exc:
            return yi, mi, case, member, None, None, exc


    def _native_eli_output_is_current(path: Path, expected: dict) -> bool:
        if not path.exists():
            return False
        try:
            with xr.open_dataset(path, decode_times=False) as cached:
                return all(cached.attrs.get(key) == value for key, value in expected.items())
        except Exception:
            return False


    for case_key, case_info in E3SM_CASES.items():
        data_dir = Path(case_info["data_dir"])
        case_prefix = case_info["case_prefix"]
        cache_tag = case_info["cache_tag"]
        display_name = case_info.get("display_name", case_key)
        outdir = OUTDIR_BY_E3SM_CASE[case_key]

        print(f"\n############################################################")
        print(f"# E3SM case group: {case_key} ({display_name})")
        print(f"# data_dir: {data_dir}")
        print(f"# outdir  : {outdir}")
        print(f"############################################################")

        for init_month in INIT_MONTHS:
            outfile_mon = outdir / f"E3SMLE{init_month:02d}_ELI_native_N{OUT_NENS:02d}_M{NLEAD:02d}.nc"
            outfile_seas = outdir / f"E3SMLE{init_month:02d}_ELI_native_N{OUT_NENS:02d}_M{NLEAD:02d}_seas.nc"
            target_months = (init_month - 1 + np.arange(NLEAD)) % 12 + 1
            seasonal_lead_indices = np.flatnonzero(np.isin(target_months, [1, 4, 7, 10]))
            seasonal_leads = seasonal_lead_indices.astype(np.int32) + 1

            expected_common = {
                "case_prefix": case_prefix,
                "init_month": int(init_month),
                "start_year": int(START_YEAR),
                "end_year": int(END_YEAR),
                "run_nens": int(RUN_NENS),
                "eli_lat_min": ELI_LAT_MIN,
                "eli_lat_max": ELI_LAT_MAX,
                "eli_lon_min": ELI_LON_MIN,
                "eli_lon_max": ELI_LON_MAX,
                "tc_lat_half": TC_LAT_HALF,
                "sst_variable": OCN_SST_VAR,
                "sst_preprocessing_version": int(SST_PREPROCESSING_VERSION),
                "eli_native_algorithm_version": 2,
            }
            expected_mon = {**expected_common, "frequency": "monthly"}
            expected_seas = {**expected_common, "frequency": "seasonal"}
            outputs_current = (
                _native_eli_output_is_current(outfile_mon, expected_mon)
                and _native_eli_output_is_current(outfile_seas, expected_seas)
            )
            if not FORCE_REWRITE and outputs_current:
                print(f"Skipping {case_key} init month {init_month:02d} - compatible monthly and seasonal outputs exist.")
                continue
            if not FORCE_REWRITE and (outfile_mon.exists() or outfile_seas.exists()):
                print("Regenerating incomplete, stale, or incompatible native ELI outputs.")

            print(f"\n=== {case_key} init month {init_month:02d} ===")
            t0 = time.time()

            eli_all = np.full(
                (len(lead_years), NLEAD, RUN_NENS),
                np.nan, dtype=np.float32,
            )
            eli_all_seas = np.full(
                (len(lead_years), len(seasonal_leads), RUN_NENS),
                np.nan, dtype=np.float32,
            )

            cases_for_month = [
                f"{case_prefix}_{year}{init_month:02d}0100" for year in lead_years
            ]

            # Keep this serial unless CONFIG["eli_workers"] is intentionally raised.
            # Concurrent netCDF/HDF5 reads can crash with native lock errors.
            workers = min(int(NWORKERS), RUN_NENS)

            for yi, (year, case) in enumerate(zip(lead_years, cases_for_month)):
                case_dir = data_dir / case
                if not case_dir.is_dir():
                    print(f"  [WARN] Case directory not found: {case_dir}")
                    continue

                tasks = [
                    (
                        yi, mi, case, member,
                        case_dir / member / "archive" / "ocn" / "hist",
                        seasonal_lead_indices,
                    )
                    for mi, member in enumerate(members_avail)
                ]

                if workers == 1:
                    results_iter = map(_compute_case_member, tasks)
                    pool = None
                else:
                    pool = ThreadPoolExecutor(max_workers=workers)
                    futures = [pool.submit(_compute_case_member, task) for task in tasks]
                    results_iter = (future.result() for future in as_completed(futures))

                try:
                    for _, mi, case_name, member, monthly, seasonal, exc in results_iter:
                        if exc is not None:
                            print(f"  [ERROR] {case_name}/{member}: {exc}")
                            continue
                        eli_all[yi, :, mi] = monthly
                        eli_all_seas[yi, :, mi] = seasonal
                finally:
                    if pool is not None:
                        pool.shutdown(wait=True)

                n_ok = int(np.sum(np.isfinite(eli_all[yi])))
                print(f"  [{yi + 1:3d}/{len(lead_years)}] {case}  - {n_ok}/{NLEAD * RUN_NENS} valid values")

            # ---- build output dataset ----
            ds_out = xr.Dataset(
                {
                    "eli": xr.DataArray(
                        eli_all,
                        dims=("Y", "L", "M"),
                        coords={
                            "Y": np.array(lead_years, dtype=np.int32),
                            "L": np.arange(1, NLEAD + 1, dtype=np.int32),
                            "M": np.arange(RUN_NENS, dtype=np.int32),
                        },
                        attrs={
                            "long_name": "Equatorial Longitude Index",
                            "units": "degrees_east",
                            "description": (
                                "Area-weighted centroid longitude of warm SST cells "
                                "(SST > tropical-mean SST) in the equatorial Pacific "
                                f"(lat {ELI_LAT_MIN}-{ELI_LAT_MAX} deg, "
                                f"lon {ELI_LON_MIN}-{ELI_LON_MAX} deg).  "
                                f"Tc reference band: +/-{TC_LAT_HALF} deg."
                            ),
                        },
                    )
                }
            )

            ds_out["Y"].attrs = {"long_name": "initialization year", "units": "year"}
            ds_out["L"].attrs = {"long_name": "forecast lead month", "units": "months"}
            ds_out["M"].attrs = {"long_name": "ensemble member index"}
            time_values = np.array([
                [
                    cftime.DatetimeNoLeap(
                        year + (init_month - 1 + lead_idx) // 12,
                        (init_month - 1 + lead_idx) % 12 + 1,
                        15,
                    )
                    for lead_idx in range(NLEAD)
                ]
                for year in lead_years
            ], dtype=object)
            ds_out["time"] = xr.DataArray(time_values, dims=("Y", "L"))
            ds_out["time"].attrs = {"long_name": "forecast valid time"}

            ds_out["member_id"] = xr.DataArray(
                np.array(members_avail, dtype="U5"),
                dims="M",
                attrs={"long_name": "ensemble member label"},
            )

            ds_out_seas = ds_out.isel(L=seasonal_lead_indices).copy()
            ds_out_seas["eli"] = xr.DataArray(
                eli_all_seas,
                dims=("Y", "L", "M"),
                coords={
                    "Y": ds_out["Y"],
                    "L": seasonal_leads,
                    "M": ds_out["M"],
                },
                attrs=ds_out["eli"].attrs,
            )

            ds_out.attrs = {
                "case_key":     case_key,
                "display_name": display_name,
                "cache_tag":    cache_tag,
                "case_prefix":  case_prefix,
                "data_dir":     str(data_dir),
                "init_month":   int(init_month),
                "start_year":   int(START_YEAR),
                "end_year":     int(END_YEAR),
                "case_nens":    int(CASE_NENS),
                "run_nens":     int(RUN_NENS),
                "nworkers":     int(workers),
                "eli_lat_min":  ELI_LAT_MIN,
                "eli_lat_max":  ELI_LAT_MAX,
                "eli_lon_min":  ELI_LON_MIN,
                "eli_lon_max":  ELI_LON_MAX,
                "tc_lat_half":  TC_LAT_HALF,
                "sst_preprocessing_version": int(SST_PREPROCESSING_VERSION),
                "eli_native_algorithm_version": 2,
                "sst_units": "degC",
                "sst_variable": OCN_SST_VAR,
                "source_grid": "native MPAS-Ocean",
                "frequency": "monthly",
            }
            ds_out_seas.attrs = {
                **ds_out.attrs,
                "frequency": "seasonal",
                "seasonal_aggregation": (
                    "Three-month mean native SST centered on Jan/Apr/Jul/Oct, "
                    "followed by ELI centroid calculation"
                ),
            }

            # Atomic writes
            for dataset, outfile in ((ds_out, outfile_mon), (ds_out_seas, outfile_seas)):
                tmp_file = outfile.with_suffix(".tmp.nc")
                try:
                    dataset.to_netcdf(
                        tmp_file,
                        encoding={"eli": {"zlib": True, "complevel": 1, "dtype": "float32"}},
                    )
                    os.replace(tmp_file, outfile)
                finally:
                    if tmp_file.exists():
                        tmp_file.unlink()

            elapsed = time.time() - t0
            mon_nan = int(np.isnan(eli_all).sum())
            seas_nan = int(np.isnan(eli_all_seas).sum())
            print(f"  Saved monthly  -> {outfile_mon} ({mon_nan}/{eli_all.size} NaN values)")
            print(f"  Saved seasonal -> {outfile_seas} ({seas_nan}/{eli_all_seas.size} NaN values)")
            print(f"  Elapsed: {elapsed:.0f}s")



############################################################
# E3SM case group: E3SM-FOSIRL (E3SMv3-FOSIRL)
# data_dir: /global/cfs/cdirs/e3smdata/simulations/S2S2D
# outdir  : /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/JRA55_FOSIRL/sst_index/timeseries
############################################################

=== E3SM-FOSIRL init month 05 ===
  [  1/39] WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1980050100  - 240/240 valid values
  [  2/39] WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1981050100  - 240/240 valid values
  [  3/39] WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1982050100  - 240/240 valid values
  [  4/39] WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1983050100  - 240/240 valid values
  [  5/39] WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1984050100  - 240/240 valid values
  [  6/39] WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1985050100  - 240/240 valid values
  [  7/39] WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL_1986050100  - 240

### Verify ELI Outputs

In [ ]:
PROCESS_REGRIDDED_ELI = (
    CONFIG.get("process_eli", True)
    and ELI_INPUT_GRID in {"regridded", "both"}
)

if not PROCESS_NATIVE_ELI and not PROCESS_REGRIDDED_ELI:
    print("E3SM ELI is disabled; skipping verification.")
else:
    import matplotlib.pyplot as plt

    output_nens = int(CONFIG["e3sm_nens"])
    for case_key, outdir in OUTDIR_BY_E3SM_CASE.items():
        print(f"\n=== {case_key}: {outdir} ===")
        for init_month in INIT_MONTHS:
            products = []
            if PROCESS_NATIVE_ELI:
                native_stem = f"E3SMLE{init_month:02d}_ELI_native_N{output_nens:02d}_M{NLEAD:02d}"
                products.extend([
                    ("native", "monthly", outdir / f"{native_stem}.nc"),
                    ("native", "seasonal", outdir / f"{native_stem}_seas.nc"),
                ])
            if PROCESS_REGRIDDED_ELI:
                regridded_stem = f"E3SMLE{init_month:02d}_ELI_N{output_nens:02d}_M{NLEAD:02d}"
                products.extend([
                    ("regridded", "monthly", outdir / f"{regridded_stem}.nc"),
                    ("regridded", "seasonal", outdir / f"{regridded_stem}_seas.nc"),
                ])

            for grid_kind, frequency, outfile in products:
                if not outfile.exists():
                    print(f"[MISSING] {grid_kind} {frequency}: {outfile}")
                    continue

                with xr.open_dataset(outfile) as ds:
                    eli = ds["eli"]
                    n_valid = int(np.isfinite(eli).sum())
                    n_total = int(eli.size)
                    print(f"\n{grid_kind} {frequency}, init {init_month:02d} -> {outfile.name}")
                    print(f"  dims   : {dict(eli.sizes)}")
                    if n_valid:
                        print(f"  min    : {float(eli.min(skipna=True)):.2f} degE")
                        print(f"  max    : {float(eli.max(skipna=True)):.2f} degE")
                        print(f"  mean   : {float(eli.mean(skipna=True)):.2f} degE")
                    else:
                        print("  min/max/mean: unavailable (all values are NaN)")
                    print(f"  valid  : {n_valid}/{n_total}")

                    if frequency == "monthly" and n_valid:
                        eli_yr0 = eli.isel(Y=0).values
                        lead_values = eli["L"].values
                        fig, ax = plt.subplots(figsize=(9, 3))
                        ax.plot(lead_values, eli_yr0, color="gray", linewidth=0.7, alpha=0.6)
                        ax.plot(lead_values, np.nanmean(eli_yr0, axis=-1),
                                color="k", linewidth=2, label="Ens mean")
                        ax.set_xlabel("Lead month")
                        ax.set_ylabel("ELI (degE)")
                        ax.set_title(
                            f"{case_key} {grid_kind} ELI - init {init_month:02d}, "
                            f"year {int(ds['Y'].isel(Y=0))}"
                        )
                        ax.legend(fontsize=9)
                        plt.tight_layout()
                        plt.show()
